In [1]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
import logging
import math
from torch.optim.lr_scheduler import ExponentialLR
from src.language_models.dictionary_corpus import Corpus, Dictionary
from src.language_models.utils import batchify, load_model, get_batch
import torch.optim as optim


In [76]:
class CBR_RNN(nn.Module):
    # goal here is to reuse CBR_RNN but with scaled dot product attention for more efficient computations.
    # Also I got rid of options such as loading pretrained embeddings, and ablating attention to simplify the code.
    # In the future if those options are needed, they can still be copy pasted from William's code as the structure hasn't changed
    def __init__(self, ntoken, ninp, nhid, nheads, bptt, dropout=0.5, device=None):
        super().__init__()
        # same layers as Timkey
        self.device = device
        self.nheads = nheads
        self.tanh = nn.Tanh()
        self.drop = nn.Dropout(dropout)
        self.score_attn = nn.Softmax(dim=-1)
        self.encoder = nn.Embedding(ntoken, ninp)
        #self.pos_encoder = PositionalEncoding(ninp, bptt)
        self.q = nn.Linear(ninp + nhid, nhid)
        self.intermediate_h = nn.Linear(nhid * 4, nhid * 4)
        self.decoder = nn.Linear(nhid, ntoken)
        self.q_norm = torch.nn.LayerNorm(nhid)
        self.int_norm = torch.nn.LayerNorm(nhid * 4)
        self.f_norm = torch.nn.LayerNorm(nhid * 3)
        self.nhid = nhid
        self.final_h = nn.Linear(nhid * 4, nhid * 3)
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=nhid, num_heads=nheads, batch_first=True
        )

        self.init_weights()

    def init_weights(self):
        """Initialize model weights for better training dynamics"""
        # General initialization
        for name, param in self.named_parameters():
            if "weight" in name:
                if "norm" in name:
                    nn.init.ones_(param)
                elif "encoder" in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                elif "decoder" in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                else:
                    # Standard He initialization for processing layers
                    nn.init.kaiming_normal_(param, mode="fan_in", nonlinearity="tanh")
            elif "bias" in name:
                nn.init.zeros_(param)

    def init_cache(self, observation, nheads):
        """Initialize hidden state and attention caches with better initialization strategy"""
        if len(observation.size()) > 1:
            bsz = observation.size(dim=-1)
        else:
            bsz = 1

        hidden = torch.zeros(1, bsz, self.nhid).to(self.device) 
        if nheads == 1:
            key_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) 
            value_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) 
        elif nheads>1:
            key_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) 
            value_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) 
        return hidden, key_cache, value_cache


    def update_cache(self, key_cache, value_cache, hidden, key_cache_i, value_cache_i, hidden_i, nheads):
        hidden_i = hidden_i.unsqueeze(0)
        hidden = torch.cat((hidden, hidden_i), dim=0)
        if nheads == 1:
                key_cache_i = key_cache_i.unsqueeze(1).unsqueeze(1)
                value_cache_i = value_cache_i.unsqueeze(1).unsqueeze(1)
                key_cache = torch.cat((key_cache, key_cache_i), dim=2)
                value_cache = torch.cat((value_cache, value_cache_i), dim=2)
        else:
            key_cache_i = key_cache_i.unsqueeze(1)
            value_cache_i = value_cache_i.unsqueeze(1)
            key_cache = torch.cat((key_cache, key_cache_i), dim=1)
            value_cache = torch.cat((value_cache, value_cache_i), dim=1)
            
        return key_cache, value_cache, hidden
    
    def hard_attention_single_head(self,query, key, value, temperature, gumbel_softmax=None, attn_mask=None,dropout_p=0.0,
        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:
        L, S = query.size(-2), key.size(-2)
        scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        if is_causal:
            assert attn_mask is None
            temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
            attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
            attn_bias.to(query.dtype)

        if attn_mask is not None:
            if attn_mask.dtype == torch.bool:
                attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
            else:
                attn_bias = attn_mask + attn_bias

        if enable_gqa:
            key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)
            value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)

        attn_weight = query @ key.transpose(-2, -1) * scale_factor
        attn_weight += attn_bias
        if gumbel_softmax: 
            attn_weight = torch.gumbel_softmax(attn_weight, tau=temperature, hard=False, dim=-1)
        else : 
            attn_weight = attn_weight/temperature
            attn_weight = torch.softmax(attn_weight, dim=-1)
        attn_weight = torch.dropout(attn_weight, dropout_p, train=self.training)
        return attn_weight @ value
    
    def hard_attention_multi_head(self,query, key, value, num_heads, temperature, 
                                attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, 
                                enable_gqa=False) -> torch.Tensor:

      
        batch_size, seq_len, embed_dim = query.shape
        head_dim = embed_dim // num_heads
        
        # Reshape for multi-head attention: (batch_size, seq_len, num_heads, head_dim)
        query = query.view(batch_size, seq_len, num_heads, head_dim)
        key = key.view(batch_size, key.size(1), num_heads, head_dim)
        value = value.view(batch_size, value.size(1), num_heads, head_dim)
        
        # Transpose to: (batch_size, num_heads, seq_len, head_dim)
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)
        
        # Get dimensions for attention computation
        L, S = query.size(-2), key.size(-2)
        scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
        
        # Create attention bias
        attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        
        # Apply causal masking if needed
        if is_causal:
            assert attn_mask is None
            temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
            attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
            attn_bias = attn_bias.to(query.dtype)
        
        # Apply custom attention mask if provided
        if attn_mask is not None:
            if attn_mask.dtype == torch.bool:
                attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
            else:
                attn_bias = attn_mask + attn_bias
        
        # Handle grouped query attention
        if enable_gqa:
            key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)
            value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)
        
        # Compute attention weights for all heads
        # query: (batch_size, num_heads, seq_len, head_dim)
        # key: (batch_size, num_heads, seq_len, head_dim)
        print(query.shape)
        print(key.shape)
        attn_weight = query @ key.transpose(-2, -1) * scale_factor
        attn_weight += attn_bias.unsqueeze(0).unsqueeze(0)  # Broadcast to all heads
        
       
        attn_weight = torch.nn.functional.gumbel_softmax(attn_weight, tau=temperature, hard=False, dim=-1)
        
        
        # Apply dropout
        attn_weight = torch.dropout(attn_weight, dropout_p, train=self.training)
        
        # Apply attention to values
        # attn_weight: (batch_size, num_heads, seq_len, seq_len)
        # value: (batch_size, num_heads, seq_len, head_dim)
        attn_output = attn_weight @ value  # (batch_size, num_heads, seq_len, head_dim)
        
        # Reshape back to original format
        attn_output = attn_output.transpose(1, 2)  # (batch_size, seq_len, num_heads, head_dim)
        attn_output = attn_output.contiguous().view(batch_size, seq_len, embed_dim)
        
        return attn_output
        
    def attention_layer(self, query, key_cache, value_cache, nheads, temperature, gumbel_softmax):
        if nheads == 1 :
                query = query.unsqueeze(1)
                
                # Ensure all tensors are on the same device
                if query.device != key_cache.device:
                    key_cache = key_cache.to(query.device)
                if query.device != value_cache.device:
                    value_cache = value_cache.to(query.device)
                if gumbel_softmax is False:  
                    try:
                        attn_output = scaled_dot_product_attention(
                            query, key_cache, value_cache, is_causal=False
                        )
                    except Exception as e:
                        logging.error(f"Error in attention computation: {str(e)}")
                        raise
                    attn = attn_output.squeeze(1).squeeze(1)
                    del attn_output  # No longer needed after squeezing
                    query = query.squeeze(1).squeeze(1)
                else:
                    try:
                        attn_output = self.hard_attention_single_head(
                            query, key_cache, value_cache, temperature, is_causal=False
                        )
                    except Exception as e:
                        logging.error(f"Error in attention computation: {str(e)}")
                        raise
                    attn = attn_output.squeeze(1).squeeze(1)
                    del attn_output  # No longer needed after squeezing
                    query = query.squeeze(1).squeeze(1)

            
        else:
            if gumbel_softmax is False : 
                attn_output, _ = self.multihead_attn(
                    query, key_cache, value_cache, is_causal=False
                )
                attn = attn_output.squeeze(1)
                del attn_output  # No longer needed after squeezing
                query = query.squeeze(1)
            else : 
                attn_output = self.hard_attention_multi_head(query, key_cache, value_cache, nheads, temperature)
                attn = attn_output.squeeze(1)
                del attn_output  # No longer needed after squeezing
                query = query.squeeze(1)
            
        return attn, query
    
    def intermediate_layers(self, i, emb, query, attn, hidden):
        intermediate_input = torch.cat((emb[i], query, attn, hidden[-1]), -1)
        del query, attn  
        intermediate = self.drop(
            self.tanh(self.int_norm(self.intermediate_h(intermediate_input)))
        )
        del intermediate_input  
        final_output = self.drop(self.tanh(self.f_norm(self.final_h(intermediate))))
        del intermediate  
        key_cache_i, value_cache_i, hidden_i = final_output.split(self.nhid, dim=-1)
        del final_output
        return key_cache_i, value_cache_i, hidden_i
    
    def get_query(self, emb, hidden):
        combined = torch.cat((emb, hidden[-1]), -1)
        query = self.drop(self.tanh(self.q_norm(self.q(combined))))
        del combined  # No longer needed after creating query
        query = query.unsqueeze(1)
        return query
    
    def forward(self, observation, initial_cache, nheads, temperature, gumbel_softmax):
        seq_len = observation.size(0)
        hidden, key_cache, value_cache = initial_cache
        # 1. Encode observations
        emb = self.drop(self.encoder(observation))
        # if positional_encoding:
        #     emb=self.pos_encoder(emb)
        del observation  # No longer needed after encoding
        
        for i in range(seq_len):
            # 2. Concatenate with previous hidden state
            
            
            query = self.get_query(emb[i], hidden)
            
            attn, query = self.attention_layer(query, key_cache, value_cache, nheads, temperature, gumbel_softmax)
            
            key_cache_i, value_cache_i, hidden_i = self.intermediate_layers(i, emb, query, attn, hidden)
            
            key_cache, value_cache, hidden = self.update_cache(key_cache, value_cache, hidden, key_cache_i, value_cache_i, hidden_i, nheads)
            
            del key_cache_i, value_cache_i, hidden_i  # No longer needed after concatenation

        decoded = self.decoder(hidden[1:])

        return decoded, hidden


# Scheduler

In [7]:
#dummy optimizer to access Pytorch's API
temp_holder = torch.nn.Parameter(torch.tensor(1.0), requires_grad=False)
temp_optimizer = torch.optim.SGD([temp_holder], lr=1.0)  # lr is irrelevant


In [8]:
temp_scheduler = ExponentialLR(temp_optimizer, gamma=0.99)


In [9]:
class TemperatureScheduler:
    def __init__(self, start_temp=1.0, min_temp=0.05, decay_rate=0.95):
        self.temperature = start_temp
        self.min_temp = min_temp
        self.decay_rate = decay_rate
        self.step_count = 0

    def step(self):
        self.temperature = max(self.min_temp, self.temperature * self.decay_rate)
        self.step_count += 1
        return self.temperature

    def get_temperature(self):
        return self.temperature


In [10]:
temp_scheduler = TemperatureScheduler(start_temp=1.0, min_temp=0.05, decay_rate=0.95)


# Load data and model

In [3]:
device = torch.device('cpu')
# # Old way to load data (from colorlessgreenRNNs)
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')
ntokens = len(corpus.dictionary)

eval_batch_size = 10

# Use regular batchify for all data
train_data = batchify(corpus.train, 512, device)
val_data = batchify(corpus.valid, eval_batch_size, device)
test_data = batchify(corpus.test, eval_batch_size, device)


                                  
criterion = nn.CrossEntropyLoss()

In [77]:
model = CBR_RNN(ntokens, 650, 650, nheads=5, bptt=0.5, dropout=0.5, device=device)


In [78]:
lr=0.001
optimizer = optim.Adam(model.parameters(), lr=lr)

In [79]:

# Turn on training mode which enables dropout
model.train()
total_loss = 0


for batch, i in enumerate(range(0, train_data.size(0) - 1, 35)):
    current_temp = temp_scheduler.get_temperature()
    # Get batch
    data, targets = get_batch(train_data, i, 35)
    data, targets = data.to(device), targets.to(device)
    optimizer.zero_grad()
    

    cache = model.init_cache(data, 5)#for CBR_RNN, initialize cache once per batch as in the original code
    output,_ = model(data, cache, 5, current_temp, True)
    
    # Reshape outputs and targets
    output_flat = output.reshape(-1, output.size(-1))
    targets_flat = targets.reshape(-1)
    
    # Calculate loss
    loss = criterion(output_flat, targets_flat)
        
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

    optimizer.step() 
    temp_scheduler.step()  

    total_loss += loss.item()

        

torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 2, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 3, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 4, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 5, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 6, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 7, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 8, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 9, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 10, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 11, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 12, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 13, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 14, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 15, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 16, 130])
torch.Size([512, 5, 1, 130])
torch.Size([512, 5, 17, 130])
torch.

KeyboardInterrupt: 